# InternVL3.5-30B-A3B: experiments 1 and 2

Attach the uploaded `kaggle_new_models` dataset, enable Internet, and select **T4 x2** manually. This notebook runs only `internvl35`. Default precision is expert-NF4; TRC uses eight separately encoded frames. See README.md for methodology and limits.

Experiment 1 runs eleven conditions: the unpruned baseline plus L2 and random at 100%, 25%, 50%, 75%, and 70% retention. Both 100% conditions retain every visual token in its original order. Random uses seed 0. 


In [ ]:
from pathlib import Path
import subprocess, sys, os, shutil
matches = list(Path('/kaggle/input').rglob('kaggle_new_models/run.py'))
assert len(matches) == 1, f'Expected one uploaded package; found {matches}'
CODE = matches[0].parent
MODEL = 'internvl35'
OUTPUT = Path('/kaggle/working/internvl35_retention_sweep_results')
RETENTIONS = [1.0, 0.25, 0.50, 0.75, 0.70]
RANDOM_SEEDS = [0]
RUN_ARGS = ['--retentions', *map(str, RETENTIONS), '--random-seeds', *map(str, RANDOM_SEEDS)]
OUTPUT_ARGS = ['--output-root', str(OUTPUT)]
assert 'INTERNVL_RETENTIONS' in (CODE / 'run.py').read_text(), 'Upload the updated package with the retention sweep runner.'
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
subprocess.run(['bash', str(CODE / 'setup_kaggle.sh')], check=True)
subprocess.run([sys.executable, '-m', 'pytest', str(CODE), '-q', '-p', 'no:cacheprovider'], check=True)


## Smoke test
Two examples per experiment, in separate folders. Weight downloads are still full size, but only one shard is retained at a time. A failed native-equivalence check or incomplete matrix stops the subprocess.

Experiment 1 runs eleven conditions: the unpruned baseline plus L2 and random at 100%, 25%, 50%, 75%, and 70% retention. Both 100% conditions retain every visual token in its original order. Random uses seed 0. 


In [ ]:
subprocess.run([sys.executable, str(CODE / 'run.py'), '--model', MODEL, '--smoke'] + RUN_ARGS + OUTPUT_ARGS, check=True)
subprocess.run([sys.executable, str(CODE / 'analyze.py'), '--model', MODEL, '--smoke'] + OUTPUT_ARGS, check=True)


In [ ]:
import json, pandas as pd
for experiment in (1, 2):
    root = OUTPUT / MODEL / f'experiment_{experiment}_smoke'
    print(f'Experiment {experiment}:', json.loads((root / 'native_equivalence.json').read_text()))
    display(pd.read_csv(root / 'summaries' / 'summary.csv'))


## Full runs
After inspecting the smoke results, run this cell. Each experiment uses 256 examples. Experiment 2 runs all four frame conditions. Rerun the same cell to resume; results must still be present under `/kaggle/working`. Model loading downloads shards again because the smoke subprocess released its model. Full runtime has not been measured; save outputs before the Kaggle session expires.

Experiment 1 runs eleven conditions: the unpruned baseline plus L2 and random at 100%, 25%, 50%, 75%, and 70% retention. Both 100% conditions retain every visual token in its original order. Random uses seed 0. 


In [ ]:
subprocess.run([sys.executable, str(CODE / 'run.py'), '--model', MODEL, '--limit', '256'] + RUN_ARGS + OUTPUT_ARGS, check=True)
subprocess.run([sys.executable, str(CODE / 'analyze.py'), '--model', MODEL, '--limit', '256'] + OUTPUT_ARGS, check=True)


In [ ]:
for experiment in (1, 2):
    root = OUTPUT / MODEL / f'experiment_{experiment}_n256'
    display(pd.read_csv(root / 'summaries' / 'summary.csv'))
display(pd.read_csv(OUTPUT / MODEL / 'experiment_2_n256/summaries/condition_accuracy.csv'))


## Download outputs
This cell can also be run after an interrupted experiment to preserve partial progress. Keep raw routes, predictions, manifests and configuration with the summaries.


In [ ]:
archive = shutil.make_archive(f'/kaggle/working/{MODEL}_experiment_results', 'zip', OUTPUT)
from IPython.display import FileLink, display
display(FileLink(archive))
